# UrduStack — Live Demo Launcher (Hackathon Submission)

This notebook launches the full UrduStack playground on a free Colab T4 GPU
and gives you a **public `*.gradio.live` URL valid for 72 hours**.

**Run all 5 cells.** The trained model is committed to the repo, so no
upload is needed — cell 4 will detect the files and skip.

**What the evaluator sees:**
- Text Analysis: Urdu / Roman Urdu / code-switched text → normalized script, risk score, phrase contributions, NER entities, plain-language recommendation
- Speech Analysis: Whisper transcription + same pipeline
- Comparison: naive keyword baseline vs. UrduStack

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Run all 5 cells in order
3. After ~3–5 minutes, cell 5 prints a public URL — copy it into your submission
4. Keep this tab open — closing it kills the demo

If the URL expires, re-run this notebook to get a fresh one.

In [ ]:
import sys, torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Demo will run on CPU (slower).")
    print("For best results: Runtime > Change runtime type > T4 GPU")
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    print(f"PyTorch: {torch.__version__}")
    print("GPU check passed.")

In [ ]:
import subprocess, time, os

def pip_install(packages, attempt=1):
    print(f"Attempt {attempt}: installing packages...")
    subprocess.run(["pip", "uninstall", "-y", "torchao"], capture_output=True)
    result = subprocess.run(
        ["pip", "install", "-q", "-U"] + packages,
        capture_output=True, text=True
    )
    return result.returncode == 0

packages = [
    "transformers>=4.46.0",
    "peft>=0.13.2",
    "accelerate>=1.1.0",
    "openai-whisper>=20231117",
    "pandas",
    "gradio>=5.1.0",
    "sentencepiece>=0.2.0",
    "safetensors>=0.4.0",
    "matplotlib>=3.8.0",
    "numpy>=1.26.0",
    "soundfile>=0.12.1",
    "pydantic>=2.6.0",
    "python-multipart>=0.0.9",
    "requests>=2.31.0",
    "huggingface-hub>=0.21.0",
]

ok = pip_install(packages, 1)
if not ok:
    print("First attempt failed, retrying in 5s...")
    time.sleep(5)
    ok = pip_install(packages, 2)
if not ok:
    raise RuntimeError("pip install failed after 2 attempts")

import peft, transformers
print(f"peft={peft.__version__}  transformers={transformers.__version__}")
print("All dependencies verified.")

In [ ]:
import os, shutil, subprocess, time

if os.path.exists('UrduStack/UrduStack'):
    shutil.rmtree('UrduStack/UrduStack')
    print('Removed nested UrduStack/UrduStack')

if os.path.exists('UrduStack'):
    os.chdir('UrduStack')
    subprocess.run(['git', 'pull'])
else:
    for attempt in range(1, 4):
        result = subprocess.run(
            ['git', 'clone', 'https://github.com/munazat/UrduStack.git'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            break
        print(f"Clone failed (attempt {attempt}/3): {result.stderr.strip()}")
        if attempt < 3:
            time.sleep(5)
    else:
        raise RuntimeError("Could not clone repo after 3 attempts")
    os.chdir('UrduStack')

print(f"Working directory: {os.getcwd()}")

for f in ['playground.py', 'app/models/model_manager.py']:
    if not os.path.exists(f):
        raise RuntimeError(f"CRITICAL: missing file: {f}")

print("Repo ready.")

In [ ]:
import os, glob
from google.colab import files

model_dir = 'models/risk_lora'
os.makedirs(model_dir, exist_ok=True)
os.makedirs('models', exist_ok=True)

# Check if model already exists (from previous upload or zip)
adapter_config = os.path.join(model_dir, 'adapter_config.json')
adapter_weights = os.path.join(model_dir, 'adapter_model.safetensors')
adapter_weights_bin = os.path.join(model_dir, 'adapter_model.bin')
temp_file = 'models/temperature.txt'

has_adapter = os.path.exists(adapter_config) and (
    os.path.exists(adapter_weights) or os.path.exists(adapter_weights_bin)
)
has_temp = os.path.exists(temp_file) and open(temp_file).read().strip()

if has_adapter and has_temp:
    print("Model files already detected! Skipping upload.")
    print(f"  adapter_config.json: OK")
    print(f"  adapter weights: OK")
    print(f"  temperature.txt: {open(temp_file).read().strip()}")
else:
    print("=" * 60)
    print("UPLOAD YOUR TRAINED MODEL FILES")
    print("=" * 60)
    print()
    print("A file picker will appear. Select ALL of these files:")
    print()
    print("  From models/risk_lora/:")
    print("    - adapter_config.json")
    print("    - adapter_model.safetensors (or .bin)")
    print("    - tokenizer.json")
    print("    - tokenizer_config.json")
    print("    - sentencepiece.bpe.model")
    print("    - vocab.txt (if present)")
    print("    - special_tokens_map.json (if present)")
    print()
    print("  From models/:")
    print("    - temperature.txt")
    print()
    print("You can select multiple files with Ctrl+click.")
    print()

    uploaded = files.upload()

    print(f"\nUploaded {len(uploaded)} file(s). Sorting into correct locations...")

    for filename in uploaded:
        src = filename
        basename = os.path.basename(filename)

        if basename == 'temperature.txt':
            dst = 'models/temperature.txt'
        else:
            dst = os.path.join(model_dir, basename)

        if src != dst:
            import shutil
            shutil.move(src, dst)
            print(f"  {basename} -> {dst}")
        else:
            print(f"  {basename} -> OK")

    # Verify everything is in place
    print("\nVerifying...")
    errors = []

    if not os.path.exists(os.path.join(model_dir, 'adapter_config.json')):
        errors.append("adapter_config.json not found in models/risk_lora/")

    if not os.path.exists(os.path.join(model_dir, 'adapter_model.safetensors')) and \
       not os.path.exists(os.path.join(model_dir, 'adapter_model.bin')):
        errors.append("adapter_model.safetensors or .bin not found in models/risk_lora/")

    if not os.path.exists('models/temperature.txt'):
        errors.append("temperature.txt not found in models/")
    elif not open('models/temperature.txt').read().strip():
        errors.append("temperature.txt is empty")

    if errors:
        print("\nERRORS:")
        for e in errors:
            print(f"  - {e}")
        print("\nRe-run this cell and upload the missing files.")
    else:
        print("\nAll required files in place!")
        temp_val = open('models/temperature.txt').read().strip()
        print(f"  Temperature: {temp_val}")
        adapter_files = os.listdir(model_dir)
        print(f"  Adapter files: {len(adapter_files)} files")
        for f in sorted(adapter_files):
            print(f"    - {f}")

In [ ]:
import os, sys

# Ensure we are in the repo root
os.chdir('/content/UrduStack')
sys.path.insert(0, '/content/UrduStack')

# Final pre-flight check
required = [
    'models/risk_lora/adapter_config.json',
    'models/risk_lora/adapter_model.safetensors',
    'models/temperature.txt',
    'playground.py',
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("ERROR: missing required files:")
    for f in missing:
        print(f"  - {f}")
    print("\nRe-run cell 4 (upload) to provide them.")
    sys.exit(1)

print("Pre-flight check: all required files present.")
print(f"  adapter_config.json: OK")
print(f"  adapter_model.safetensors: OK")
print(f"  temperature.txt: {open('models/temperature.txt').read().strip()}")

from playground import build_demo

print("\nBuilding unified demo...")
demo = build_demo()

print()
print("=" * 60)
print("LAUNCHING DEMO")
print("=" * 60)
print()
print("First launch downloads ~2.5 GB of base models from Hugging Face")
print("(xlm-roberta-base + Davlan NER). This takes 2-5 minutes.")
print()
print("A public *.gradio.live URL will appear below. Copy it into your")
print("submission. The URL is valid for 72 hours while this cell runs.")
print("Keep this tab open - closing it kills the demo.")
print()

demo.launch(share=True, quiet=False)